# TruFor + ViT clasificador — Colab T4

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)*.

Pipeline:

1. imagen → **preprocesamiento** (`src/preprocess.py`, variantes antes de TruFor)
2. **TruFor** (congelado) → score global + mapa de manipulación + mapa de confianza
3. **normalización de tamaño** de los mapas a `MAP_SIZE`×`MAP_SIZE` (`src/maps.py`) y cache
4. **ViT desde cero** (RGB + mapa + confianza) → probabilidad de manipulación

Lo que hay que conservar entre sesiones vive en Drive (`BASE`); el disco de Colab
se borra. Las celdas de **extracción** y **entrenamiento** son *reanudables*: si la
sesión se corta, volvé a correr 1→4, la de empaquetado y la que se interrumpió.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "Cambiá el entorno de ejecución a GPU (T4)"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

REPO_URL = "https://github.com/Valentin190306/Deepfake_Detection_PDI_2026.git"
BASE = "/content/drive/MyDrive/pdi"          # datos, cache y checkpoints (persistente)

VARIANT = "raw"                               # variantes en src/preprocess.py (ej. log05)
MAP_SIZE = 384                                # lado S de entrada del ViT
MAX_SIDE = 1280                               # límite de memoria para TruFor en T4
CACHE_KEY = f"{VARIANT}__s{MAP_SIZE}__ms{MAX_SIDE}"
PACKED = f"/content/packed/{VARIANT}__s{MAP_SIZE}"   # local: se regenera en ~2 min
print(CACHE_KEY, "->", PACKED)

## 1. Repo, dependencias y modelo TruFor

No se reinstala `torch`: Colab ya trae una build con CUDA y reinstalarlo suele
romper la GPU. `requirements.txt` fija `timm==0.9.16` porque TruFor usa la API
vieja de timm.

In [ ]:
import os
os.makedirs(BASE, exist_ok=True)
!test -d /content/repo || git clone -q {REPO_URL} /content/repo
%cd /content/repo
!git pull -q --ff-only || echo "(pull no aplicable)"
!git log --oneline -1
!python --version
!pip install -q -r requirements.txt

In [ ]:
!bash scripts/get_trufor.sh

## 2. Verificación del entorno

Tiene que dar ~`0.19` para las *pristine* y ~`0.99` para las *tampered* (en GPU los
decimales varían un poco). Si no coincide, no sigas: algo del entorno está mal.

In [ ]:
!python src/detect.py third_party/trufor/images/ --variant raw --max-side 0

## 3. Datos: IMD2020 desde Kaggle

Dataset `saikotahmed/imd2020`: 2010 manipuladas con máscara + 414 auténticas.
Si la descarga anónima falla, creá en **Secretos** de Colab (ícono de llave)
`KAGGLE_USERNAME` y `KAGGLE_KEY` y habilitá el acceso al notebook.

La preparación copia las imágenes a `{BASE}/data` **una sola vez** y arma el
`manifest.csv` con etiqueta y **grupo de escena**. El grupo es lo que evita que
un original quede en train y su versión manipulada en validación.

In [ ]:
import os, kagglehub
try:
    imd_path = kagglehub.dataset_download("saikotahmed/imd2020")
except Exception as e:
    print("Descarga anónima falló:", e)
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    imd_path = kagglehub.dataset_download("saikotahmed/imd2020")
print(imd_path)

### 3b. Diagnóstico antes de escribir

`--dry-run` muestra cuántos pares imagen–máscara encontró, ejemplos de nombres y
cuántos grupos quedan tras fusionar originales con manipuladas. Si el
emparejamiento es <50% o los grupos son raros, no sigas: revisá el layout del
dataset antes de copiar 600 MB.

`--pairs balanced` deja sólo las escenas que tienen **auténtica y manipulada** y
repite la auténtica una vez por variante, para que el conjunto quede 1:1. En
IMD2020: **1552 filas, 1042 imágenes únicas, 264 escenas** (antes 2424/2424/1606).
Se descarta el 61% de las manipuladas, pero cada escena pesa igual en las dos
clases y el modelo no puede aprender "esta foto es trucada" por el fondo.

Si ya corriste el pipeline sin `--pairs`, el cache tiene las 2424 imágenes: la
extracción saltea las 1042 que ya están y `pack_cache` empaqueta las 2424
igual (no rompe nada, sólo ocupa ~1.8 GB). Para recuperar el espacio, borrá
`{BASE}/cache/{CACHE_KEY}` y volvé a extraer.

In [ ]:
!python src/prepare_imd2020.py --src {imd_path} --out {BASE}/data --prefix imd_ --group-by content --pairs balanced --dry-run

In [ ]:
!python src/prepare_imd2020.py --src {imd_path} --out {BASE}/data --prefix imd_ --group-by content --pairs balanced
# El manifest pareado cambia el conjunto de escenas (1606 -> 264): el split
# viejo no las conoce y apply_splits falla a propósito. Se regenera solo.
!rm -f {BASE}/runs/splits.json

In [ ]:
import pandas as pd
m = pd.read_csv(f"{BASE}/data/manifest.csv")
print(m.groupby("source").size().to_string())
print("grupos:", m.group.nunique())
display(m.head())

## 4. Extracción de mapas TruFor (reanudable)

Una pasada por imagen: variante → TruFor → normalización a `MAP_SIZE` → `.npz` en
Drive. Se saltea lo ya procesado, así que si se corta se vuelve a lanzar.

En T4, con `MAX_SIDE=1280`, tarda ~25–30 min para las 1042 imágenes del manifest
pareado (~30–60 min para las 2424 del completo). `MAX_SIDE`
evita OOM con las fotos de ~3000×4400 px; TruFor trabaja a resolución nativa y el
ViT de todos modos recibe 384×384.

In [ ]:
!python src/extract_maps.py --manifest {BASE}/data/manifest.csv --cache {BASE}/cache --variants {VARIANT} --map-size {MAP_SIZE} --max-side {MAX_SIDE}

## 5. Empaquetado local del cache

Leer 2400 `.npz` desde Drive en cada época es lento y frágil. Este paso los junta
en un único `.npy` memmap en el disco local de Colab (~1.8 GB, ~2 min). **Hay que
repetirlo al inicio de cada sesión** (o después de agregar imágenes).

In [ ]:
!python src/pack_cache.py --cache {BASE}/cache/{CACHE_KEY} --out {PACKED} --overwrite

## 6. Prueba rápida (3 épocas, 20 pasos)

Sirve para confirmar que baja la loss y que el split tiene las dos clases. No
mires las métricas: son de juguete.

In [ ]:
!python src/train.py --manifest {BASE}/data/manifest.csv --packed {PACKED} --splits {BASE}/runs/splits.json --out {BASE}/runs/smoke --size {MAP_SIZE} --epochs 3 --limit-steps 20 --patience 0

## 7. Entrenamiento completo (reanudable)

Checkpoints en `{BASE}/runs/vit/`: `last.pt` (cada época), `best.pt` (mejor AUC de
validación), `config.json` y `log.csv`. Si Colab se desconecta, volvé a correr esta
celda: retoma desde `last.pt`.

El ViT es **desde cero** y arranca igual al baseline de TruFor (logit residual con
`alpha=1`); a partir de ahí aprende una corrección. `--patience 20` corta si deja
de mejorar.

La **loss auxiliar** (`--aux-weight`, default 0.5) supervisa también la rama sin
el prior. Sin ella el residual tiene una salida trivial: dejar el ViT en cero y
sólo atenuar el score (medido en el primer run: Spearman −0.96 contra TruFor y
AUC idéntico al baseline). Para el test limpio, `--no-score --no-residual`.

Si cambiás de estrategia, empezá con `--fresh` o un `--out` nuevo: `best.pt`
sólo se sobrescribe si mejora el AUC de la corrida guardada.

La validación pareada tiene **61 auténticas únicas** (61 escenas): el AUC se
mueve ±0.05 con cualquier cambio, y `best.pt` se elige por AUC, así que una
diferencia chica entre corridas puede ser suerte. Para el número final, repetí
con 3 semillas (`--seed 1`, `--seed 2`, `--resplit`) y reportá media ± desvío.

In [ ]:
!python src/train.py --manifest {BASE}/data/manifest.csv --packed {PACKED} --splits {BASE}/runs/splits.json --out {BASE}/runs/vit --size {MAP_SIZE} --epochs 150 --bs 16 --workers 2

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
log = pd.read_csv(f"{BASE}/runs/vit/log.csv")
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
for c in ["acc", "bal_acc", "f1"]:
    ax[0].plot(log.epoch, log[c], label=c)
ax[0].set_title("clasificación"); ax[0].legend(); ax[0].grid(alpha=.3)
for c in ["auc", "ap"]:
    ax[1].plot(log.epoch, log[c], label=c)
ax[1].set_title("ranking"); ax[1].legend(); ax[1].grid(alpha=.3)
ax[2].plot(log.epoch, log.train_loss, label="train")
ax[2].plot(log.epoch, log.val_loss, label="val")
ax[2].set_title("loss"); ax[2].legend(); ax[2].grid(alpha=.3)
plt.tight_layout(); plt.show()

### 7b. ¿El ViT aporta algo sobre TruFor?

Compara el ranking de la rama propia (`vit_logit`, sin el prior) contra el del
score de TruFor. Si `AUC vit solo` es ~0.5 y el spearman es ~−1, la rama sólo
está calibrando el score. Con `--aux-weight 0.5` (default) el AUC del ViT debería
superar 0.5; `--no-score --no-residual` es el test limpio, sin atajo.

In [ ]:
# ¿El ViT aporta señal propia o sólo copia el score de TruFor?
import json, sys, numpy as np, torch
from pathlib import Path
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr
sys.path.insert(0, "src")
from dataset import read_manifest, apply_splits, dedupe_rows, MapDataset
from model import build_model

ck = torch.load(f"{BASE}/runs/vit/best.pt", map_location="cpu", weights_only=False)
cfg = ck["cfg"]
model = build_model(**cfg).eval(); model.load_state_dict(ck["model"])
_, va = apply_splits(read_manifest(Path(f"{BASE}/data/manifest.csv")),
                     json.load(open(f"{BASE}/runs/splits.json")))
ds = MapDataset(packed=PACKED, rows=dedupe_rows(va),
                channels="maps+rgb" if cfg["in_chans"] == 5 else "maps")

alpha = model.alpha.item() if model.residual else 0.0
vit, prior, ys = [], [], []
with torch.no_grad():
    for x, y, s in DataLoader(ds, batch_size=32):
        p = torch.log(s.clamp(1e-4, 1 - 1e-4) / (1 - s.clamp(1e-4, 1 - 1e-4)))
        out = model(x, s)
        vit.append((out - alpha * p).numpy()); prior.append(p.numpy()); ys.append(y.numpy())
vit, prior, ys = map(np.concatenate, (vit, prior, ys))
print(f"alpha {alpha:.3f} | std(vit) {vit.std():.3f} vs std(prior) {prior.std():.3f}")
print(f"AUC vit solo {roc_auc_score(ys, vit):.3f} | AUC prior {roc_auc_score(ys, prior):.3f} | "
      f"AUC final {roc_auc_score(ys, vit + alpha * prior):.3f}")
print(f"spearman(vit, prior) {spearmanr(vit, prior).statistic:.3f}")

## 8. Evaluación

Métricas sobre la validación de IMD2020, matriz de confusión, ROC/PR y una
galería con los errores más confiados.

> ⚠️ **TruFor se entrenó con IMD2020**, así que estas métricas están infladas.
> El número honesto es el de un test externo (ver la última sección).

In [ ]:
!python src/eval.py --manifest {BASE}/data/manifest.csv --packed {PACKED} --ckpt {BASE}/runs/vit/best.pt --splits {BASE}/runs/splits.json --out {BASE}/runs/eval --split val --gallery 6 --bootstrap 500

In [ ]:
from IPython.display import Image as Img, display
display(Img(filename=f"{BASE}/runs/eval/metricas.png"))
display(Img(filename=f"{BASE}/runs/eval/galeria.png"))

## 9. Test externo (recomendado)

Para reportar un número honesto hace falta un conjunto que **no** sea IMD2020
(por ejemplo CASIA, Columbia, CoMoFoD o fotos propias manipuladas). El flujo es
el mismo: armar un `manifest.csv` con las columnas `file, mask, label, group,
source, width, height`, correr `extract_maps.py` + `pack_cache.py` sobre él y
evaluar con `--split all`.

```bash
python src/extract_maps.py --manifest /ruta/externo/manifest.csv     --cache {BASE}/cache_ext --variants raw --map-size 384 --max-side 1280
python src/pack_cache.py --cache {BASE}/cache_ext/raw__s384__ms1280 --out /content/packed/ext
python src/eval.py --manifest /ruta/externo/manifest.csv --packed /content/packed/ext     --ckpt {BASE}/runs/vit/best.pt --out {BASE}/runs/eval_ext --split all
```

## 10. Probar el modelo con tus propias imágenes

Subí fotos desde tu PC y el notebook corre el pipeline completo sobre ellas:
variante → TruFor → normalización a `MAP_SIZE` → ViT. Muestra el score de TruFor,
la probabilidad del ViT, el veredicto y los mapas.

- La **probabilidad del ViT** es la decisión (el umbral se eligió en validación).
- El **score de TruFor** va como referencia: es el baseline que el ViT corrige.
- Todo se procesa con la **misma configuración del cache** con el que se entrenó
  (variante, tamaño y modos de mapa se leen de `_meta.json`), así que la entrada
  es idéntica a la del entrenamiento.

> ⚠️ Una probabilidad baja **no demuestra** que la foto sea auténtica, ni una alta
> que esté manipulada. TruFor se entrenó con IMD2020 y reacciona fuerte a fotos
> recomprimidas, capturas de pantalla o con filtros. Usalo como señal, no como
> veredicto legal.


In [ ]:
# Subí imágenes desde tu PC -> /content/mis_imagenes (disco de Colab: se borra al
# cerrar la sesión). Volvé a correr esta celda para agregar más.
import os
from pathlib import Path
from google.colab import files

UPLOADS = Path("/content/mis_imagenes")
UPLOADS.mkdir(parents=True, exist_ok=True)
os.chdir(UPLOADS)                  # files.upload() escribe en el directorio actual
print("Elegí una o varias imágenes (JPG, PNG, WEBP, BMP):")
files.upload()
os.chdir("/content/repo")      # las demás celdas usan rutas relativas al repo

EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}
paths = sorted(p for p in UPLOADS.iterdir() if p.suffix.lower() in EXTS)
print(f"\n{len(paths)} imágenes en {UPLOADS}")
for p in paths:
    print("  ", p.name)
# para empezar de cero:  import shutil; shutil.rmtree(UPLOADS)


In [ ]:
# Corre TruFor + el ViT sobre cada imagen subida y muestra el veredicto.
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

sys.path.insert(0, "/content/repo/src")
from dataset import CHANNELS, MEAN, STD          # misma normalización que el dataset
from detect import analyze, load_model
from maps import normalize_maps, resize_rgb, to_uint8
from model import build_model
from preprocess import apply_variant
from preprocess import max_side as limit_side

CKPT = f"{BASE}/runs/vit/best.pt"
assert Path(CKPT).exists(), f"no existe {CKPT}; entrená primero (sección 7)"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Configuración con la que se generó el cache: así la entrada es la misma que
# vio el ViT durante el entrenamiento.
meta = json.loads((Path(BASE) / "cache" / CACHE_KEY / "_meta.json").read_text())
S = int(meta["map_size"])
VARIANT, HEAT, CONF = meta["variant"], meta["heat_mode"], meta["conf_mode"]
MAX_SIDE_USED = int(meta.get("max_side") or 0)

ck = torch.load(CKPT, map_location=device, weights_only=False)
model = build_model(**ck["cfg"]).to(device).eval()
model.load_state_dict(ck["model"])          # sin esto el ViT predice con pesos al azar
channels = "maps+rgb" if ck["cfg"]["in_chans"] == 5 else "maps"
c0, c1 = CHANNELS[channels]
thr = float(ck.get("thr", 0.5))
trufor, weights = load_model(device)
print(f"checkpoint época {ck.get('epoch')} | TruFor {weights.name} | {S}px | "
      f"canales={channels} | umbral {thr:.3f}")


def predict_one(path):
    """imagen -> probabilidad del ViT + mapas a S px (igual que extract_maps.py)."""
    arr = np.array(Image.open(path).convert("RGB"))
    if MAX_SIDE_USED:
        arr = limit_side(arr, MAX_SIDE_USED)
    arr = apply_variant(arr, VARIANT)
    t = torch.tensor(arr.transpose(2, 0, 1), dtype=torch.float) / 256.0
    score, heat, conf = analyze(trufor, t.unsqueeze(0), device)

    m = normalize_maps(heat, conf, S, HEAT, CONF)
    # Se pasa por uint8 igual que el cache: la entrada queda byte a byte igual.
    u8 = np.concatenate([resize_rgb(arr, S).transpose(2, 0, 1),
                         to_uint8(m[0])[None], to_uint8(m[1])[None]], 0)
    x = torch.from_numpy(u8).float() / 255.0
    x = torch.cat([(x[:3] - MEAN) / STD, x[3:]], 0)[c0:c1].unsqueeze(0)
    with torch.no_grad():
        prob = torch.sigmoid(model(x.to(device),
                                   torch.tensor([score], device=device))).item()
    return dict(prob=prob, score=score, rgb=u8[:3].transpose(1, 2, 0),
                heat=m[0], conf=m[1])


paths = sorted(p for p in Path("/content/mis_imagenes").iterdir()
               if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp", ".bmp",
                                       ".tif", ".tiff"})
assert paths, "no hay imágenes: corré la celda de arriba y subí alguna"

res = sorted(({**predict_one(p), "name": p.name} for p in paths),
             key=lambda r: -r["prob"])
print(f"\n{'archivo':<30}{'TruFor':>8}{'ViT':>8}   veredicto")
for r in res:
    print(f"{r['name'][:30]:<30}{r['score']:>8.3f}{r['prob']:>8.3f}   "
          f"{'MANIPULADA' if r['prob'] >= thr else 'auténtica'}")

for r in res:
    veredicto = "MANIPULADA" if r["prob"] >= thr else "auténtica"
    fig, ax = plt.subplots(1, 4, figsize=(17, 4.5))
    ax[0].imshow(r["rgb"])
    ax[0].set_title(r["name"][:40], fontsize=9)
    ax[1].imshow(r["heat"], cmap="RdBu_r", vmin=0, vmax=1)
    ax[1].set_title("mapa de manipulación")
    ax[2].imshow(r["conf"], cmap="viridis", vmin=0, vmax=1)
    ax[2].set_title("confianza")
    ax[3].imshow(r["rgb"])
    ax[3].imshow(r["heat"], cmap="RdBu_r", vmin=0, vmax=1, alpha=0.45)
    ax[3].set_title(f"overlay | ViT {r['prob']:.3f} ({veredicto})")
    for a in ax:
        a.axis("off")
    plt.tight_layout()
    plt.show()
